# CAD parameter validation

Small demo: load **Cameo** `requirements.json` and **CATIA** `catia_parameters.json`, compare bounded requirements to CAD values, show an HTML report.

Prerequisites: `istari_labs_helpers` kernel, `samples/.env` — see [Chaining jobs](../chaining_jobs.ipynb).

## 1 · Connect

In [ ]:
from pathlib import Path

from IPython.display import HTML, Markdown, display
from istari_labs_helpers import IstariPlatform, JobDefinition

from validation_lib import fetch_artifact, load_json, render_html, run

NOTEBOOK_DIR = Path.cwd()
platform = IstariPlatform.from_env(str(NOTEBOOK_DIR.parent / ".env"))
print(platform)

## 2 · Job IDs

Paste completed extraction job IDs, or set `USE_LOCAL_FILES = True` for bundled JSON in this folder.

In [ ]:
# CAMEO_EXTRACTION_JOB_ID = "c26fe0a2-c428-4195-b72d-acd8cd3c0b0a"
CAMEO_EXTRACTION_JOB_ID = "7d1d08bd-c1ec-4cc2-8ae4-39c5d058699a"
CATIA_EXTRACTION_JOB_ID = "adea6f83-122b-4966-9b01-226988e78531"

USE_LOCAL_FILES = False
REQUIREMENTS_FILE = "requirements.json"
PARAMETERS_FILE = "parameters.json"

## 3 · Retrieve extraction outputs

- **Requirements** — Cameo extraction (`requirements.json`)
- **Parameters** — CATIA extraction (`catia_parameters.json`)

In [ ]:
def read_istari_artifact(job_id: str, name: str):
    return fetch_artifact(platform, job_id, name)

if USE_LOCAL_FILES:
    requirements = load_json(NOTEBOOK_DIR / REQUIREMENTS_FILE)
    parameters = load_json(NOTEBOOK_DIR / PARAMETERS_FILE)
else:
    requirements = read_istari_artifact(CAMEO_EXTRACTION_JOB_ID, REQUIREMENTS_FILE)
    parameters = read_istari_artifact(CATIA_EXTRACTION_JOB_ID, PARAMETERS_FILE)

print(f"Loaded {REQUIREMENTS_FILE} and {PARAMETERS_FILE}")

## 4 · Generate validation report

In [ ]:
rows = run(requirements, parameters)
html = render_html(rows)

(NOTEBOOK_DIR / "validation_report.html").write_text(html, encoding="utf-8")
print(f"{len(rows)} comparable checks — all passed" if rows and all(r.passed for r in rows) else f"{len(rows)} checks")
display(HTML(html))

## 5 · Update wing length on CATIA model

Run `@istari:update_parameters` on the CATIA assembly. Set `CATIA_MODEL_ID` to your model resource UUID from the platform (Models tab). Requires a **dassault_catia_v5** agent on Windows.

In [ ]:
CATIA_MODEL_ID = "paste-your-catia-model-uuid-here"

WING_LENGTH_PARAM = r"SA ISTARI_ONE\WING.2\WING_LENGTH"
WING_LENGTH_MM = 1600  # mm — choose a value within your requirement bounds

In [ ]:
catia_model = platform.get_model(CATIA_MODEL_ID)

update_def = JobDefinition(
    function="@istari:update_parameters",
    tool_name="dassault_catia_v5",
    tool_version="6R2023",
    operating_system="Windows 10",
    input_json_data={
        "parameters": {
            WING_LENGTH_PARAM: WING_LENGTH_MM,
        },
    },
)

update_job = catia_model.submit_job(update_def)
print(f"Submitted update job {update_job.id}; polling...")

update_job.wait(
    timeout=900,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"WING_LENGTH set to {WING_LENGTH_MM} mm — job {update_job.id} completed")